# Unsupervised Learning — Hands-On Lab
## K-Means Clustering | PCA | Market Basket Analysis

**Course:** AI Engineering — Unsupervised Learning Module  
**Focus:** Implementing and comparing three cornerstone unsupervised techniques.

---


In [ ]:
# ============================================================
# CELL 1: Environment Setup & Imports
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import warnings
warnings.filterwarnings('ignore')

# Visual settings
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("All libraries imported successfully!")


## Part 1: K-Means Clustering — Finding Natural Groups 

### Learning Objectives 

* **Understand the iterative K-Means algorithm** — K-Means অ্যালগরিদম কীভাবে ধাপে ধাপে বা ইটারেটিভ উপায়ে (Iterative process) কাজ করে তা গভীরভাবে বোঝা।
* **Visualize cluster assignments and centroids** — ক্লাস্টার অ্যাসাইনমেন্ট এবং সেন্ট্রয়েডগুলোর (Centroids - সেন্ট্রাল পয়েন্ট) অবস্থান ও মুভমেন্ট চাক্ষুষ বা ভিজ্যুয়ালাইজ করা।
* **Use the Elbow Method and Silhouette Score to choose K** — আমাদের ডেটার জন্য ক্লাস্টারের সংখ্যা বা $K$-এর মান ঠিক কত হবে, তা এলবো মেথড (Elbow Method) এবং সিলুয়েট স্কোরের (Silhouette Score) মাধ্যমে বৈজ্ঞানিক উপায়ে নির্ধারণ করা।
* **Compare K-Means performance on different data shapes** — ভিন্ন ভিন্ন আকৃতির বা শেপের ডেটাসেটের (Data shapes) ওপর K-Means অ্যালগরিদম কেমন পারফর্ম করে তা তুলনা এবং বিশ্লেষণ করা।

---

In [ ]:
# ============================================================
# CELL 2: K-Means — Basic Clustering on Blob Data
# ============================================================

# Generate synthetic data with 4 known clusters
X_blobs, y_true = make_blobs(
    n_samples=500, n_features=2, centers=4,
    cluster_std=1.0, random_state=42
)

# Scale features (K-Means is distance-based)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_blobs)

# Apply K-Means
kmeans = KMeans(n_clusters=4, init='k-means++', n_init=10, random_state=42)
labels = kmeans.fit_predict(X_scaled)
centroids = kmeans.cluster_centers_

# Evaluation metrics
sil_score = silhouette_score(X_scaled, labels)
ch_score = calinski_harabasz_score(X_scaled, labels)

print("=== K-MEANS RESULTS ===")
print(f"Inertia (WCSS): {kmeans.inertia_:.2f}")
print(f"Silhouette Score: {sil_score:.3f}")
print(f"Calinski-Harabasz: {ch_score:.1f}")
print(f"Iterations to converge: {kmeans.n_iter_}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ground truth
axes[0].scatter(X_scaled[:, 0], X_scaled[:, 1], c=y_true, cmap='viridis', s=50, alpha=0.7)
axes[0].set_title('Ground Truth (4 Blobs)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

# K-Means result
scatter = axes[1].scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels, cmap='viridis', s=50, alpha=0.7, edgecolors='k')
axes[1].scatter(centroids[:, 0], centroids[:, 1], c='red', marker='X', s=200, edgecolors='black', linewidths=2, label='Centroids')
axes[1].set_title('K-Means Clustering (K=4)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# CELL 3: Elbow Method — Finding the Optimal K
# ============================================================

k_range = range(1, 16)
inertias = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    if k > 1:
        silhouettes.append(silhouette_score(X_scaled, km.labels_))
    else:
        silhouettes.append(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow plot
axes[0].plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=4, color='red', linestyle='--', alpha=0.7, label='Optimal K = 4')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Silhouette plot
axes[1].plot(k_range[1:], silhouettes[1:], 'go-', linewidth=2, markersize=8)
axes[1].axvline(x=4, color='red', linestyle='--', alpha=0.7, label='Optimal K = 4')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Elbow at K=4: WCSS stops decreasing significantly after this point.")
print(f"Silhouette peak at K=4: Score = {silhouettes[4]:.3f} (close to +1 = well-separated)")


In [ ]:
# ============================================================
# CELL 4: K-Means Limitations — Non-Convex Shapes
# ============================================================

# Generate moon-shaped data (non-convex clusters)
X_moons, y_moons = make_moons(n_samples=300, noise=0.1, random_state=42)
X_moons_scaled = StandardScaler().fit_transform(X_moons)

# K-Means on moon data
kmeans_moons = KMeans(n_clusters=2, init='k-means++', n_init=10, random_state=42)
labels_moons = kmeans_moons.fit_predict(X_moons_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ground truth
axes[0].scatter(X_moons_scaled[:, 0], X_moons_scaled[:, 1], c=y_moons, cmap='coolwarm', s=50, alpha=0.7)
axes[0].set_title('Ground Truth: Moon Shapes', fontsize=13, fontweight='bold')

# K-Means result
axes[1].scatter(X_moons_scaled[:, 0], X_moons_scaled[:, 1], c=labels_moons, cmap='coolwarm', s=50, alpha=0.7, edgecolors='k')
axes[1].scatter(kmeans_moons.cluster_centers_[:, 0], kmeans_moons.cluster_centers_[:, 1],
                c='black', marker='X', s=200, edgecolors='white', linewidths=2, label='Centroids')
axes[1].set_title('K-Means on Moon Shapes (FAILS)', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

print("K-Means assumes clusters are convex (round) and isotropic.")
print("Moon shapes are non-convex — no single centroid can capture each moon.")
print("SOLUTION: Use DBSCAN or spectral clustering for non-convex shapes.")


## Part 2: Principal Component Analysis (PCA) 

### Learning Objectives 

* **Understand eigenvalue decomposition and variance capture** — আইগেনভ্যালু ডিকম্পোজিশন (Eigenvalue decomposition) কী এবং এটি কীভাবে ডেটার ভেতরের মূল বৈচিত্র্য বা ভ্যারিয়েন্স (Variance capture) খুঁজে বের করে তা গভীরভাবে বোঝা।
* **Visualize explained variance and cumulative variance** — প্রতিটি প্রিন্সিপাল কম্পোনেন্ট কতটুকু ইনফরমেশন ধরে রাখছে (Explained variance) এবং ক্রমান্বয়ে তা মোট কত হচ্ছে (Cumulative variance), তা গ্রাফের মাধ্যমে ভিজ্যুয়ালাইজ করা।
* **Use PCA for dimensionality reduction and noise removal** — ডেটার পারফরম্যান্স ধরে রেখে তার মাত্রা বা ফিচার কমানো (Dimensionality reduction) এবং অপ্রয়োজনীয় নয়েজ দূর করার (Noise removal) প্র্যাক্টিক্যাল ব্যবহার শেখা।
* **Interpret principal component loadings** — মূল ফিচারগুলোর সাথে প্রিন্সিপাল কম্পোনেন্টগুলোর রিলেশন বা লোডিংস (Component loadings) দেখে ডেটার অভ্যন্তরীণ বিন্যাস ব্যাখ্যা করা।



In [ ]:
# ============================================================
# CELL 5: PCA — Dimensionality Reduction & Variance Analysis
# ============================================================

# Generate high-dimensional data with structure
np.random.seed(42)
n_samples = 500
n_features = 10

# Create correlated features
X_high = np.random.randn(n_samples, n_features)
# Make features 1-3 correlated, 4-6 correlated, 7-10 noise
X_high[:, 1] = X_high[:, 0] + np.random.randn(n_samples) * 0.3
X_high[:, 2] = X_high[:, 0] + np.random.randn(n_samples) * 0.3
X_high[:, 4] = X_high[:, 3] + np.random.randn(n_samples) * 0.3
X_high[:, 5] = X_high[:, 3] + np.random.randn(n_samples) * 0.3

# Scale
X_high_scaled = StandardScaler().fit_transform(X_high)

# Apply PCA
pca = PCA()
X_pca_full = pca.fit_transform(X_high_scaled)

explained_var = pca.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

print("=== PCA RESULTS ===")
print(f"Total features: {n_features}")
print("Explained Variance by Component:")
for i, (ev, cv) in enumerate(zip(explained_var, cumulative_var)):
    print(f"  PC{i+1}: {ev:.3f} (cumulative: {cv:.3f})")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar + line plot
axes[0].bar(range(1, len(explained_var)+1), explained_var, alpha=0.7, color='steelblue', label='Individual')
axes[0].plot(range(1, len(cumulative_var)+1), cumulative_var, 'ro-', linewidth=2, markersize=6, label='Cumulative')
axes[0].axhline(y=0.95, color='green', linestyle='--', alpha=0.7, label='95% Threshold')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('PCA Explained Variance', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Scree plot
axes[1].plot(range(1, len(explained_var)+1), pca.explained_variance_, 'bo-', linewidth=2, markersize=8)
axes[1].axhline(y=1, color='red', linestyle='--', alpha=0.7, label='Kaiser Criterion (lambda=1)')
axes[1].set_xlabel('Principal Component')
axes[1].set_ylabel('Eigenvalue')
axes[1].set_title('Scree Plot', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# CELL 6: PCA — 2D Projection & Component Loadings
# ============================================================

# Reduce to 2D for visualization
pca_2d = PCA(n_components=2)
X_pca = pca_2d.fit_transform(X_high_scaled)

# Component loadings (how much each original feature contributes)
loadings = pd.DataFrame(
    pca_2d.components_.T,
    columns=['PC1', 'PC2'],
    index=[f'Feature_{i+1}' for i in range(n_features)]
)
print("Component Loadings:")
print(loadings.round(3))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 2D projection
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.6, s=50, c='steelblue', edgecolors='k')
axes[0].set_xlabel(f'PC1 ({explained_var[0]:.1%} variance)')
axes[0].set_ylabel(f'PC2 ({explained_var[1]:.1%} variance)')
axes[0].set_title('Data in First Two Principal Components', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Loadings heatmap
sns.heatmap(loadings, annot=True, cmap='RdBu_r', center=0, ax=axes[1], cbar_kws={'label': 'Loading'})
axes[1].set_title('Feature Loadings on PC1 & PC2', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Original Features')

plt.tight_layout()
plt.show()

print("PC1 is dominated by Features 1, 4 (correlated pairs).")
print("PC2 is dominated by Features 2, 5 (another correlated pair).")
print("Features 8-10 (noise) have near-zero loadings — they add no structure.")


In [ ]:
# ============================================================
# CELL 7: PCA — Image Compression Demo
# ============================================================

# Create a synthetic "image" (gradient pattern)
img_size = 64
x = np.linspace(-3, 3, img_size)
y = np.linspace(-3, 3, img_size)
X_grid, Y_grid = np.meshgrid(x, y)
img = np.sin(X_grid) * np.cos(Y_grid) + 0.5 * np.sin(2*X_grid + 1)

# Flatten image for PCA
img_flat = img.reshape(img_size, -1)

# Try different numbers of components
components_to_try = [5, 10, 20, 50]
fig, axes = plt.subplots(1, len(components_to_try) + 1, figsize=(16, 3))

# Original
axes[0].imshow(img, cmap='gray')
axes[0].set_title('Original
(64x64 = 4096 pixels)')
axes[0].axis('off')

for idx, n_comp in enumerate(components_to_try):
    pca_img = PCA(n_components=n_comp)
    transformed = pca_img.fit_transform(img_flat)
    reconstructed = pca_img.inverse_transform(transformed)

    mse = np.mean((img_flat - reconstructed) ** 2)
    compression_ratio = (img_size * img_size) / (img_size * n_comp + n_comp)

    axes[idx + 1].imshow(reconstructed, cmap='gray')
    axes[idx + 1].set_title(f'{n_comp} PCs
MSE: {mse:.4f}
Ratio: {compression_ratio:.1f}x')
    axes[idx + 1].axis('off')

plt.suptitle('PCA Image Compression', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("With just 20 components, the image is recognizable (20x compression).")
print("With 50 components, nearly indistinguishable from original (1.3x compression).")
print("Trade-off: Fewer components = more compression, more loss.")


## Part 3: Market Basket Analysis — Association Rule Mining 

### Learning Objectives 

* **Understand the Apriori algorithm and its pruning strategy** — অ্যাপ্রিয়রি (Apriori) অ্যালগরিদম কীভাবে কাজ করে এবং এটি কীভাবে অপ্রয়োজনীয় কম্বিনেশনগুলো ছাঁটাই বা প্রুনিং (Pruning strategy) করে কম্পিউটেশনাল এফিশিয়েন্সি বাড়ায় তা গভীরভাবে বোঝা।
* **Calculate and interpret support, confidence, and lift** — অ্যাসোসিয়েশন রুল মাইনিংয়ের ৩টি স্তম্ভ: সাপোর্ট (Support), কনফিডেন্স (Confidence) এবং লিফট (Lift) গাণিতিকভাবে ক্যালকুলেট করা এবং এর ব্যবসায়িক আউটপুট ব্যাখ্যা করা।
* **Mine actionable rules from transaction data** — হাজার হাজার ট্রানজেকশন ডেটা (Transaction data) অ্যানালিসিস করে এমন সব কার্যকর নিয়ম বা রুলস বের করা যা সরাসরি ব্যবসায়িক সিদ্ধান্ত বা সেলস বাড়াতে সাহায্য করে।
* **Visualize rule landscapes** — খুঁজে পাওয়া রুলস বা প্যাটার্নগুলোকে বিভিন্ন ইন্টারেক্টিভ চার্ট ও গ্রাফের মাধ্যমে ভিজ্যুয়ালাইজ (Visualize rule landscapes) করা, যেন কোন প্রোডাক্টগুলোর কম্বিনেশন সবচেয়ে স্ট্রং তা এক নজরেই বোঝা যায়।


In [ ]:
# ============================================================
# CELL 8: Market Basket Analysis — Apriori Algorithm
# ============================================================

# Synthetic transaction data
transactions = [
    ['Milk', 'Bread', 'Butter', 'Eggs'],
    ['Milk', 'Bread', 'Butter'],
    ['Milk', 'Bread'],
    ['Milk', 'Butter'],
    ['Bread', 'Butter'],
    ['Milk', 'Bread', 'Butter', 'Cheese'],
    ['Beer', 'Chips'],
    ['Beer', 'Chips', 'Nuts'],
    ['Beer', 'Nuts'],
    ['Chips', 'Soda'],
    ['Milk', 'Bread', 'Butter', 'Eggs', 'Cheese'],
    ['Coffee', 'Milk', 'Sugar'],
    ['Coffee', 'Sugar'],
    ['Milk', 'Bread', 'Butter'],
    ['Beer', 'Chips'],
    ['Milk', 'Eggs'],
    ['Bread', 'Butter', 'Jam'],
    ['Coffee', 'Milk', 'Sugar', 'Cake'],
    ['Beer', 'Chips', 'Soda'],
    ['Milk', 'Bread', 'Eggs'],
]

# Encode transactions to binary matrix
te = TransactionEncoder()
te_array = te.fit_transform(transactions)
df_transactions = pd.DataFrame(te_array, columns=te.columns_)

print("Transaction Matrix (first 5 rows):")
print(df_transactions.head())
print(f"\nShape: {df_transactions.shape}")
print(f"Items: {list(te.columns_)}")

# Find frequent itemsets
frequent_itemsets = apriori(df_transactions, min_support=0.15, use_colnames=True, max_len=3)
print(f"\nFrequent Itemsets (min_support=0.15): {len(frequent_itemsets)} found")
print(frequent_itemsets.head(10))


In [ ]:
# ============================================================
# CELL 9: Association Rules — Support, Confidence, Lift
# ============================================================

# Generate rules from frequent itemsets
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)
rules = rules[rules['lift'] >= 1.0]  # Only positive associations
rules = rules.sort_values('lift', ascending=False)

print(f"=== ASSOCIATION RULES ({len(rules)} found) ===")
print("\nTop 10 Rules by Lift:")
display_cols = ['antecedents', 'consequents', 'support', 'confidence', 'lift', 'conviction']
print(rules[display_cols].head(10).to_string())

# Detailed explanation of top rule
if not rules.empty:
    top_rule = rules.iloc[0]
    ant = ', '.join(list(top_rule['antecedents']))
    con = ', '.join(list(top_rule['consequents']))
    print(f"\n=== TOP RULE ANALYSIS ===")
    print(f"Rule: {{{ant}}} -> {{{con}}}")
    print(f"Support: {top_rule['support']:.3f} ({top_rule['support']*len(transactions):.0f} out of {len(transactions)} transactions)")
    print(f"Confidence: {top_rule['confidence']:.3f} ({top_rule['confidence']*100:.1f}% of {{{ant}}} buyers also buy {{{con}}}")
    print(f"Lift: {top_rule['lift']:.3f} ({{{con}}} buyers are {top_rule['lift']:.2f}x more likely when {{{ant}}} is purchased)")


In [ ]:
# ============================================================
# CELL 10: Association Rules — Visualization
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: Support vs Confidence, colored by Lift
scatter = axes[0].scatter(
    rules['support'], rules['confidence'],
    s=rules['lift'] * 100,  # Size proportional to lift
    c=rules['lift'], cmap='viridis',
    alpha=0.7, edgecolors='black', linewidth=0.5
)
axes[0].set_xlabel('Support')
axes[0].set_ylabel('Confidence')
axes[0].set_title('Association Rules Landscape', fontsize=13, fontweight='bold')
plt.colorbar(scatter, ax=axes[0], label='Lift')
axes[0].grid(True, alpha=0.3)

# Bar chart: Top rules by lift
top_10 = rules.nlargest(10, 'lift')
rule_labels = [f"{', '.join(list(a))} -> {', '.join(list(c))}" 
               for a, c in zip(top_10['antecedents'], top_10['consequents'])]
axes[1].barh(range(len(rule_labels)), top_10['lift'], color='coral')
axes[1].set_yticks(range(len(rule_labels)))
axes[1].set_yticklabels(rule_labels, fontsize=9)
axes[1].set_xlabel('Lift')
axes[1].set_title('Top 10 Rules by Lift', fontsize=13, fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print("\n=== INTERPRETATION ===")
print("- Top-left quadrant: High confidence, low support (niche but strong rules)")
print("- Bottom-right quadrant: High support, low confidence (common but weak rules)")
print("- Color intensity: Lift > 1.5 = strong association worth acting on")


## Part 4: Integrated Pipeline — PCA + K-Means + MBA

Let's combine all three techniques on a realistic customer dataset.

---


In [ ]:
# ============================================================
# CELL 11: Integrated Pipeline — Customer Segmentation
# ============================================================

# Generate realistic customer data
np.random.seed(42)
n_customers = 500

# Create customer features with hidden segments
customers = pd.DataFrame({
    'purchase_frequency': np.concatenate([
        np.random.poisson(2, 100),   # Budget shoppers
        np.random.poisson(5, 150),   # Regular customers
        np.random.poisson(3, 100),   # Premium buyers
        np.random.poisson(1, 50),    # Bulk purchasers
        np.random.poisson(8, 100),   # Loyal members
    ]),
    'avg_order_value': np.concatenate([
        np.random.normal(30, 10, 100),
        np.random.normal(100, 30, 150),
        np.random.normal(400, 100, 100),
        np.random.normal(500, 150, 50),
        np.random.normal(200, 50, 100),
    ]),
    'online_ratio': np.concatenate([
        np.random.beta(2, 5, 100),
        np.random.beta(3, 3, 150),
        np.random.beta(5, 2, 100),
        np.random.beta(2, 5, 50),
        np.random.beta(4, 2, 100),
    ]),
    'satisfaction': np.concatenate([
        np.random.normal(3.0, 0.5, 100),
        np.random.normal(3.5, 0.6, 150),
        np.random.normal(4.2, 0.4, 100),
        np.random.normal(3.8, 0.7, 50),
        np.random.normal(4.5, 0.3, 100),
    ]),
    'discount_usage': np.concatenate([
        np.random.beta(5, 2, 100),
        np.random.beta(3, 3, 150),
        np.random.beta(2, 5, 100),
        np.random.beta(4, 2, 50),
        np.random.beta(2, 4, 100),
    ]),
})

# Scale features
features = customers[['purchase_frequency', 'avg_order_value', 'online_ratio', 'satisfaction', 'discount_usage']]
scaler = StandardScaler()
X_customers = scaler.fit_transform(features)

# Step 1: PCA for visualization
pca_cust = PCA(n_components=2)
X_pca_cust = pca_cust.fit_transform(X_customers)

# Step 2: K-Means clustering
kmeans_cust = KMeans(n_clusters=5, init='k-means++', n_init=10, random_state=42)
labels_cust = kmeans_cust.fit_predict(X_customers)

# Step 3: Profile clusters
customers['cluster'] = labels_cust
cluster_profiles = customers.groupby('cluster').mean().round(2)

print("=== CUSTOMER SEGMENTATION RESULTS ===")
print(f"Silhouette Score: {silhouette_score(X_customers, labels_cust):.3f}")
print("\nCluster Profiles (Mean Values):")
print(cluster_profiles)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PCA projection with clusters
scatter = axes[0].scatter(X_pca_cust[:, 0], X_pca_cust[:, 1], c=labels_cust, cmap='tab10', s=50, alpha=0.7, edgecolors='k')
axes[0].set_xlabel(f'PC1 ({pca_cust.explained_variance_ratio_[0]:.1%} variance)')
axes[0].set_ylabel(f'PC2 ({pca_cust.explained_variance_ratio_[1]:.1%} variance)')
axes[0].set_title('Customer Clusters in PCA Space', fontsize=13, fontweight='bold')
plt.colorbar(scatter, ax=axes[0], label='Cluster')

# Cluster profile heatmap
sns.heatmap(cluster_profiles.T, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[1], cbar_kws={'label': 'Value'})
axes[1].set_title('Cluster Feature Profiles', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Cluster')

plt.tight_layout()
plt.show()

print("\n=== CLUSTER INTERPRETATION ===")
print("Cluster 0: Low frequency, low value, high discount usage -> BUDGET SHOPPERS")
print("Cluster 1: Moderate frequency, moderate value -> REGULAR CUSTOMERS")
print("Cluster 2: Low frequency, high value, high online ratio -> PREMIUM BUYERS")
print("Cluster 3: Very low frequency, very high value -> BULK PURCHASERS")
print("Cluster 4: High frequency, high satisfaction -> LOYAL MEMBERS")
